In [23]:
# Day 2 — Step 1: Load raw train/dev/test splits and verify alignment

from pathlib import Path

DATA_DIR = Path("../data")  # adjust if your notebook isn't one level below project root

SPLITS = {
    "train": ("opus.ar-en-train.en", "opus.ar-en-train.ar"),
    "dev":   ("opus.ar-en-dev.en",   "opus.ar-en-dev.ar"),
    "test":  ("opus.ar-en-test.en",  "opus.ar-en-test.ar"),
}

def load_split(en_filename, ar_filename):
    en_path = DATA_DIR / en_filename
    ar_path = DATA_DIR / ar_filename

    with open(en_path, encoding="utf-8") as f:
        en_lines = [line.rstrip("\n") for line in f]
    with open(ar_path, encoding="utf-8") as f:
        ar_lines = [line.rstrip("\n") for line in f]

    assert len(en_lines) == len(ar_lines), (
        f"Mismatch: {en_filename} has {len(en_lines)} lines, "
        f"{ar_filename} has {len(ar_lines)} lines"
    )

    return en_lines, ar_lines

data = {}
for split_name, (en_file, ar_file) in SPLITS.items():
    en_lines, ar_lines = load_split(en_file, ar_file)
    data[split_name] = (en_lines, ar_lines)
    print(f"{split_name}: {len(en_lines)} pairs loaded, lengths match ✓")

print("\nSample from train:")
print("EN:", data["train"][0][0])
print("AR:", data["train"][1][0])

train: 1000000 pairs loaded, lengths match ✓
dev: 2000 pairs loaded, lengths match ✓
test: 2000 pairs loaded, lengths match ✓

Sample from train:
EN: And this?
AR: و هذه؟


In [24]:
# Day 2 — Step 2: Clean train split (empty lines, duplicates)

def clean_pairs(en_lines, ar_lines):
    seen = set()
    clean_en, clean_ar = [], []
    n_empty = 0
    n_dupe = 0

    for en, ar in zip(en_lines, ar_lines):
        en_s, ar_s = en.strip(), ar.strip()

        if not en_s or not ar_s:
            n_empty += 1
            continue

        key = (en_s, ar_s)
        if key in seen:
            n_dupe += 1
            continue
        seen.add(key)

        clean_en.append(en_s)
        clean_ar.append(ar_s)

    return clean_en, clean_ar, n_empty, n_dupe

train_en, train_ar = data["train"]
clean_train_en, clean_train_ar, n_empty, n_dupe = clean_pairs(train_en, train_ar)

assert len(clean_train_en) == len(clean_train_ar)

print(f"Original train pairs: {len(train_en)}")
print(f"Empty pairs removed:  {n_empty}")
print(f"Duplicate pairs removed: {n_dupe}")
print(f"Remaining clean pairs: {len(clean_train_en)}")

Original train pairs: 1000000
Empty pairs removed:  0
Duplicate pairs removed: 40071
Remaining clean pairs: 959929


In [25]:
# Day 2 — Step 3: Inspect sentence length distribution (word-level, both sides)

import numpy as np

def word_lengths(lines):
    return np.array([len(line.split()) for line in lines])

en_lens = word_lengths(clean_train_en)
ar_lens = word_lengths(clean_train_ar)

def summarize(name, lens):
    print(f"{name} word-length stats:")
    print(f"  min:    {lens.min()}")
    print(f"  max:    {lens.max()}")
    print(f"  mean:   {lens.mean():.2f}")
    print(f"  median: {np.median(lens):.1f}")
    print(f"  p95:    {np.percentile(lens, 95):.1f}")
    print(f"  p99:    {np.percentile(lens, 99):.1f}")
    print()

summarize("EN", en_lens)
summarize("AR", ar_lens)

# how many pairs have very short or very long sides?
print("Pairs with EN length == 1:", (en_lens == 1).sum())
print("Pairs with AR length == 1:", (ar_lens == 1).sum())
print("Pairs with EN length > 100:", (en_lens > 100).sum())
print("Pairs with AR length > 100:", (ar_lens > 100).sum())

EN word-length stats:
  min:    1
  max:    5277
  mean:   10.46
  median: 7.0
  p95:    33.0
  p99:    58.0

AR word-length stats:
  min:    1
  max:    5910
  mean:   8.83
  median: 6.0
  p95:    29.0
  p99:    52.0

Pairs with EN length == 1: 35031
Pairs with AR length == 1: 44425
Pairs with EN length > 100: 1215
Pairs with AR length > 100: 850


In [26]:
# Day 2 — Step 3b: Inspect length outliers before choosing cutoffs

# a) Longest examples — are these real sentences or corpus artifacts?
long_idx = np.argsort(en_lens)[-5:][::-1]
print("=== Longest EN examples ===")
for i in long_idx:
    print(f"[{en_lens[i]} words] {clean_train_en[i][:200]}...")
    print()

# b) Length-1 examples — legitimate short utterances or noise?
short_idx = np.where(en_lens == 1)[0][:10]
print("=== Sample of EN length==1 pairs ===")
for i in short_idx:
    print(f"EN: {clean_train_en[i]!r}  |  AR: {clean_train_ar[i]!r}")

# c) Length RATIO check — misaligned pairs often show a skewed en/ar ratio
ratio = np.maximum(en_lens, ar_lens) / np.maximum(np.minimum(en_lens, ar_lens), 1)
print(f"\nPairs with length ratio > 3: {(ratio > 3).sum()}")
worst_ratio_idx = np.argsort(ratio)[-5:][::-1]
print("=== Worst length-ratio examples ===")
for i in worst_ratio_idx:
    print(f"ratio={ratio[i]:.1f}  EN[{en_lens[i]}]: {clean_train_en[i][:100]}")
    print(f"           AR[{ar_lens[i]}]: {clean_train_ar[i][:100]}")
    print()

=== Longest EN examples ===
[5277 words] ==3rd millennium=====21st century===:2100 - 2099 - 2098 - 2097 - 2096 - 2095 - 2094 - 2093 - 2092 - 2091:2090 - 2089 - 2088 - 2087 - 2086 - 2085 - 2084 - 2083 - 2082 - 2081:2080 - 2079 - 2078 - 2077 -...

[1944 words] Ratify the Convention on the Rights of Persons with Disabilities (CPD) (Turkey, Croatia, Argentina) and its Optional Protocol (Croatia); ratify CPD and its Optional Protocol and harmonize its national...

[1713 words] United Nations Forum on Forests First session 11-22 June 2001 Agenda item 7 Adoption of the report of the Forum on its first session Draft report Vice-Chairman-cum-Rapporteur: Knut Øistad (Norway) Int...

[979 words] Letter dated 26 July 2001 from the Permanent Representative of Iraq to the United Nations addressed to the Secretary-General On instructions from my Government, I have the honour to notify you of cont...

[883 words] Approves the United Nations Human Settlements Programme's Medium-term Strategic and Insti

In [27]:
# Day 2 — Step 3c: Test candidate thresholds before committing

for max_len in [80, 100, 150, 200]:
    n = ((en_lens > max_len) | (ar_lens > max_len)).sum()
    print(f"max_len={max_len}: would remove {n} pairs ({100*n/len(en_lens):.2f}%)")

print()
for max_ratio in [3, 4, 5, 6]:
    n = (ratio > max_ratio).sum()
    print(f"max_ratio={max_ratio}: would remove {n} pairs ({100*n/len(en_lens):.2f}%)")

print()
# combined effect (union of both filters)
for max_len, max_ratio in [(100, 4), (150, 5)]:
    mask_bad = (en_lens > max_len) | (ar_lens > max_len) | (ratio > max_ratio)
    print(f"max_len={max_len} + max_ratio={max_ratio}: would remove {mask_bad.sum()} pairs "
          f"({100*mask_bad.sum()/len(en_lens):.2f}%)")

max_len=80: would remove 3510 pairs (0.37%)
max_len=100: would remove 1501 pairs (0.16%)
max_len=150: would remove 352 pairs (0.04%)
max_len=200: would remove 150 pairs (0.02%)

max_ratio=3: would remove 20576 pairs (2.14%)
max_ratio=4: would remove 9564 pairs (1.00%)
max_ratio=5: would remove 5454 pairs (0.57%)
max_ratio=6: would remove 3371 pairs (0.35%)

max_len=100 + max_ratio=4: would remove 10915 pairs (1.14%)
max_len=150 + max_ratio=5: would remove 5740 pairs (0.60%)


In [28]:
# Day 2 — Step 4: Apply length + ratio filter

MAX_LEN = 100
MAX_RATIO = 4

en_lens_arr = word_lengths(clean_train_en)
ar_lens_arr = word_lengths(clean_train_ar)
ratio_arr = np.maximum(en_lens_arr, ar_lens_arr) / np.maximum(np.minimum(en_lens_arr, ar_lens_arr), 1)

keep_mask = (en_lens_arr <= MAX_LEN) & (ar_lens_arr <= MAX_LEN) & (ratio_arr <= MAX_RATIO)

filtered_en = [e for e, keep in zip(clean_train_en, keep_mask) if keep]
filtered_ar = [a for a, keep in zip(clean_train_ar, keep_mask) if keep]

assert len(filtered_en) == len(filtered_ar)

print(f"Before length/ratio filter: {len(clean_train_en)}")
print(f"Removed: {len(clean_train_en) - len(filtered_en)} "
      f"({100*(1 - len(filtered_en)/len(clean_train_en)):.2f}%)")
print(f"Remaining: {len(filtered_en)}")

Before length/ratio filter: 959929
Removed: 10915 (1.14%)
Remaining: 949014


In [29]:
# Day 2 — Step 5: Subsample training pairs (reproducible)

import random

SEED = 42
N_TRAIN_SAMPLE = 40000

random.seed(SEED)

indices = list(range(len(filtered_en)))
random.shuffle(indices)
sample_indices = indices[:N_TRAIN_SAMPLE]

train_en_sample = [filtered_en[i] for i in sample_indices]
train_ar_sample = [filtered_ar[i] for i in sample_indices]

assert len(train_en_sample) == len(train_ar_sample) == N_TRAIN_SAMPLE

print(f"Subsampled {N_TRAIN_SAMPLE} pairs from {len(filtered_en)} cleaned pairs (seed={SEED})")
print("\nSample check:")
for i in range(3):
    print(f"EN: {train_en_sample[i]}")
    print(f"AR: {train_ar_sample[i]}")
    print()

Subsampled 40000 pairs from 949014 cleaned pairs (seed=42)

Sample check:
EN: You first. That's nasty.
AR: انت اول ياشقي

EN: #You're not the boss of me now #
AR: ? أنتَلستالرئيسعليّبعدالآن?

EN: Sookie.
AR: ! سوكي



In [30]:
# Day 2 — Step 5b: Check for missing-space artifacts in Arabic subsample

def max_word_char_len(line):
    words = line.split()
    return max((len(w) for w in words), default=0)

ar_max_word_lens = [max_word_char_len(line) for line in train_ar_sample]

import numpy as np
ar_max_word_lens = np.array(ar_max_word_lens)

print("Max single-word character length per line (AR subsample):")
print(f"  median: {np.median(ar_max_word_lens):.0f}")
print(f"  p95:    {np.percentile(ar_max_word_lens, 95):.0f}")
print(f"  p99:    {np.percentile(ar_max_word_lens, 99):.0f}")
print(f"  max:    {ar_max_word_lens.max()}")

# flag lines where the longest "word" looks suspiciously long
threshold = 15
suspicious_idx = np.where(ar_max_word_lens > threshold)[0]
print(f"\nLines with a 'word' longer than {threshold} chars: {len(suspicious_idx)} "
      f"({100*len(suspicious_idx)/len(train_ar_sample):.2f}%)")

print("\nSample suspicious lines:")
for i in suspicious_idx[:8]:
    print(f"EN: {train_en_sample[i]}")
    print(f"AR: {train_ar_sample[i]}")
    print()

Max single-word character length per line (AR subsample):
  median: 7
  p95:    11
  p99:    15
  max:    100

Lines with a 'word' longer than 15 chars: 376 (0.94%)

Sample suspicious lines:
EN: #You're not the boss of me now #
AR: ? أنتَلستالرئيسعليّبعدالآن?

EN: You're mistaken
AR: {\cH62D0DD}.جميعُكم مخطئون

EN: It's like my daddy used to say:
AR: مثلماكانأبّييقول :

EN: We'll drop off the food and pay our respects and go.
AR: سيكون الأمر سريعاً، سندخللنواسيهمبتعاذينا..

EN: 51 Adopted as document FCCC/SBSTA/2008/L.17.
AR: اعتُمدت بوصفها الوثيقة FCCC/SBSTA/2008/L.17.

EN: ♪ I'm gonna do something naughty ♪
AR: ♪أناسأفعلشيئامطيع ♪

EN: As a public service, to help other abductees.
AR: كخدمةٍللعامّة، كيأُساعدالمختطفينالآخرين.

EN: Statement to the Sixth Committee, the Chairman of the Ad Hoc Committee on Jurisdictional Immunities of States and their Property (A/C.6/59/SR.13, para.
AR: () بيان أمام اللجنة السادسة، رئيس اللجنة المخصصة المعنية بحصانات الدول وممتلكاتها من الولاية القضائية،

In [31]:
# Day 2 — Step 5c: Check for leftover subtitle formatting tags

import re

tag_pattern = re.compile(r'\{\\[^}]*\}')  # matches {\cH62D0DD}, {\pos(...)}, etc.

n_with_tags = sum(1 for line in train_ar_sample if tag_pattern.search(line))
print(f"AR lines with leftover subtitle tags: {n_with_tags} "
      f"({100*n_with_tags/len(train_ar_sample):.2f}%)")

# check EN side too
n_with_tags_en = sum(1 for line in train_en_sample if tag_pattern.search(line))
print(f"EN lines with leftover subtitle tags: {n_with_tags_en}")

AR lines with leftover subtitle tags: 87 (0.22%)
EN lines with leftover subtitle tags: 4


In [32]:
# Day 2 — Step 6: Strip leftover subtitle formatting tags

import re

tag_pattern = re.compile(r'\{\\[^}]*\}')

def strip_tags(line):
    return tag_pattern.sub('', line).strip()

train_en_sample = [strip_tags(line) for line in train_en_sample]
train_ar_sample = [strip_tags(line) for line in train_ar_sample]

# re-verify: any empty lines created by stripping?
empty_after_strip = sum(
    1 for en, ar in zip(train_en_sample, train_ar_sample)
    if not en.strip() or not ar.strip()
)
print(f"Empty pairs after tag-stripping: {empty_after_strip}")

# re-check tag removal worked
remaining = sum(1 for line in train_ar_sample if tag_pattern.search(line))
print(f"AR lines still containing tags: {remaining}")

Empty pairs after tag-stripping: 3
AR lines still containing tags: 0


In [33]:
# Day 2 — Step 6b: Drop empty pairs created by tag-stripping

before = len(train_en_sample)

filtered_pairs = [
    (en, ar) for en, ar in zip(train_en_sample, train_ar_sample)
    if en.strip() and ar.strip()
]
train_en_sample = [p[0] for p in filtered_pairs]
train_ar_sample = [p[1] for p in filtered_pairs]

assert len(train_en_sample) == len(train_ar_sample)
print(f"Before: {before}, After: {len(train_en_sample)}, Dropped: {before - len(train_en_sample)}")

Before: 40000, After: 39997, Dropped: 3


In [34]:
# Day 2 — Step 6c: Apply tag-stripping to dev/test (no other filtering)

dev_en, dev_ar = data["dev"]
test_en, test_ar = data["test"]

dev_en = [strip_tags(line) for line in dev_en]
dev_ar = [strip_tags(line) for line in dev_ar]
test_en = [strip_tags(line) for line in test_en]
test_ar = [strip_tags(line) for line in test_ar]

# check for any empties created, but do NOT silently drop — dev/test size should stay
# fixed unless you deliberately decide otherwise
dev_empty = sum(1 for en, ar in zip(dev_en, dev_ar) if not en.strip() or not ar.strip())
test_empty = sum(1 for en, ar in zip(test_en, test_ar) if not en.strip() or not ar.strip())

print(f"Dev: {len(dev_en)} pairs, {dev_empty} became empty after tag-strip")
print(f"Test: {len(test_en)} pairs, {test_empty} became empty after tag-strip")

Dev: 2000 pairs, 0 became empty after tag-strip
Test: 2000 pairs, 0 became empty after tag-strip


In [35]:
# Day 2 — Final dataset statistics for notebook write-up

def word_lengths(lines):
    return np.array([len(line.split()) for line in lines])

final_en_lens = word_lengths(train_en_sample)
final_ar_lens = word_lengths(train_ar_sample)

print("=== FINAL TRAIN SUBSAMPLE ===")
print(f"Pairs: {len(train_en_sample)}")
print(f"EN word length — mean: {final_en_lens.mean():.2f}, median: {np.median(final_en_lens):.1f}, "
      f"p95: {np.percentile(final_en_lens,95):.1f}, max: {final_en_lens.max()}")
print(f"AR word length — mean: {final_ar_lens.mean():.2f}, median: {np.median(final_ar_lens):.1f}, "
      f"p95: {np.percentile(final_ar_lens,95):.1f}, max: {final_ar_lens.max()}")

dev_en_lens, dev_ar_lens = word_lengths(dev_en), word_lengths(dev_ar)
test_en_lens, test_ar_lens = word_lengths(test_en), word_lengths(test_ar)

print(f"\n=== DEV === Pairs: {len(dev_en)} | EN mean: {dev_en_lens.mean():.2f}, AR mean: {dev_ar_lens.mean():.2f}")
print(f"=== TEST === Pairs: {len(test_en)} | EN mean: {test_en_lens.mean():.2f}, AR mean: {test_ar_lens.mean():.2f}")

print("\n=== EXAMPLE PAIRS ===")
for i in [0, 1, 2]:
    print(f"EN: {train_en_sample[i]}")
    print(f"AR: {train_ar_sample[i]}")
    print()

=== FINAL TRAIN SUBSAMPLE ===
Pairs: 39997
EN word length — mean: 10.24, median: 7.0, p95: 33.0, max: 99
AR word length — mean: 8.62, median: 6.0, p95: 28.0, max: 99

=== DEV === Pairs: 2000 | EN mean: 12.30, AR mean: 94.48
=== TEST === Pairs: 2000 | EN mean: 11.06, AR mean: 9.35

=== EXAMPLE PAIRS ===
EN: You first. That's nasty.
AR: انت اول ياشقي

EN: #You're not the boss of me now #
AR: ? أنتَلستالرئيسعليّبعدالآن?

EN: Sookie.
AR: ! سوكي



In [36]:
# Day 2 — Sanity check: investigate dev AR length anomaly

dev_ar_word_lens = np.array([len(line.split()) for line in dev_ar])
dev_en_word_lens = np.array([len(line.split()) for line in dev_en])

print("DEV AR word-length stats:")
print(f"  mean: {dev_ar_word_lens.mean():.2f}, median: {np.median(dev_ar_word_lens):.1f}, "
      f"p95: {np.percentile(dev_ar_word_lens,95):.1f}, p99: {np.percentile(dev_ar_word_lens,99):.1f}, "
      f"max: {dev_ar_word_lens.max()}")

# find the worst offenders
worst_idx = np.argsort(dev_ar_word_lens)[-5:][::-1]
print("\n=== Longest AR examples in dev ===")
for i in worst_idx:
    print(f"[AR len={dev_ar_word_lens[i]}, EN len={dev_en_word_lens[i]}]")
    print(f"EN: {dev_en[i][:150]}")
    print(f"AR: {dev_ar[i][:150]}")
    print()

# how many lines are driving the mean up?
print(f"Dev pairs with AR length > 200: {(dev_ar_word_lens > 200).sum()}")
print(f"Dev pairs with AR length > 500: {(dev_ar_word_lens > 500).sum()}")

DEV AR word-length stats:
  mean: 94.48, median: 6.0, p95: 29.0, p99: 51.0, max: 171144

=== Longest AR examples in dev ===
[AR len=171144, EN len=3173]
EN: 2000 Review Conference of the Parties to the Treaty on the Non-Proliferation of Nuclear Weapons Final Document Volume II Part III Documents issued at 
AR: مؤتمر الأطراف في معاهدة عدم انتشار الأسلحة النووية لاستعراض المعاهدة عام 2000 الوثيقة الختامية المجلد الثاني الجزء الثالث الوثائق الصادرة في المؤتمر ن

[AR len=181, EN len=207]
EN: The mandate of UNMEE, as set out in paragraph 2 of Security Council resolution 1320 (2000), is as follows: (a) to monitor the cessation of hostilities
AR: 6 - تتمثل ولاية بعثة الأمم المتحدة في إثيوبيا وإريتريا، بصيغتها الواردة في الفقرة 2 من قـــرار مجلــــس الأمـــــن 1320 (2000)، فيما يلـــي: (أ) مراقب

[AR len=106, EN len=98]
EN: Calls upon the international community, relevant United Nations agencies, funds and programmes and intergovernmental and non-governmental organization
AR: 14 - تهيب بالمجتم

In [37]:
# Day 2 — Remove single corrupted dev pair (171,144-word alignment failure)

bad_idx = np.argmax(dev_ar_word_lens)
print(f"Removing dev pair at index {bad_idx}: AR len={dev_ar_word_lens[bad_idx]}, EN len={dev_en_word_lens[bad_idx]}")

dev_en = [line for i, line in enumerate(dev_en) if i != bad_idx]
dev_ar = [line for i, line in enumerate(dev_ar) if i != bad_idx]

assert len(dev_en) == len(dev_ar) == 1999

# recompute stats to confirm fix
dev_ar_lens_fixed = np.array([len(line.split()) for line in dev_ar])
dev_en_lens_fixed = np.array([len(line.split()) for line in dev_en])
print(f"Dev now: {len(dev_en)} pairs")
print(f"AR mean: {dev_ar_lens_fixed.mean():.2f} (was 94.48)")
print(f"EN mean: {dev_en_lens_fixed.mean():.2f}")

Removing dev pair at index 1066: AR len=171144, EN len=3173
Dev now: 1999 pairs
AR mean: 8.91 (was 94.48)
EN mean: 10.72


In [38]:
test_ar_lens = np.array([len(line.split()) for line in test_ar])
print(f"Test AR max word length: {test_ar_lens.max()}")
print(f"Test pairs with AR length > 200: {(test_ar_lens > 200).sum()}")

Test AR max word length: 121
Test pairs with AR length > 200: 0


In [39]:
from transformers import AutoTokenizer

# First call will auto-download tokenizer files (~5MB) from Hugging Face
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Sanity check: encode a few EN samples with source lang code
sample_idx = [0, 1, 2, 3, 4]

print("=== EN -> tokens (eng_Latn) ===")
tokenizer.src_lang = "eng_Latn"
for i in sample_idx:
    text = train_en_sample[i]
    ids = tokenizer(text, return_tensors="pt").input_ids[0]
    print(f"[{i}] words={len(text.split())} tokens={len(ids)}")
    print(f"    text: {text}")
    print(f"    ids:  {ids.tolist()}")

print("\n=== AR -> tokens (arb_Arab) ===")
tokenizer.src_lang = "arb_Arab"
for i in sample_idx:
    text = train_ar_sample[i]
    ids = tokenizer(text, return_tensors="pt").input_ids[0]
    print(f"[{i}] words={len(text.split())} tokens={len(ids)}")
    print(f"    text: {text}")
    print(f"    ids:  {ids.tolist()}")

=== EN -> tokens (eng_Latn) ===
[0] words=4 tokens=11
    text: You first. That's nasty.
    ids:  [256047, 3555, 13417, 248075, 11582, 248116, 248066, 38150, 248078, 248075, 2]
[1] words=8 tokens=13
    text: #You're not the boss of me now #
    ids:  [256047, 4179, 45151, 248116, 119, 2294, 349, 78421, 452, 239, 10643, 4179, 2]
[2] words=1 tokens=5
    text: Sookie.
    ids:  [256047, 2025, 37081, 248075, 2]
[3] words=9 tokens=14
    text: - I could love you, you know. - Faith...
    ids:  [256047, 89, 117, 14173, 13121, 1259, 248079, 1259, 6382, 248075, 89, 100340, 284, 2]
[4] words=6 tokens=9
    text: I can take care of myself.
    ids:  [256047, 117, 2125, 8598, 6247, 452, 55651, 248075, 2]

=== AR -> tokens (arb_Arab) ===
[0] words=3 tokens=6
    text: انت اول ياشقي
    ids:  [256011, 7357, 3989, 26115, 18885, 2]
[1] words=2 tokens=16
    text: ? أنتَلستالرئيسعليّبعدالآن?
    ids:  [256011, 385, 14807, 248109, 248106, 379, 26348, 72960, 248197, 2113, 248241, 248129, 14468, 21670

In [40]:
import numpy as np

def token_length_stats(texts, lang_code, tokenizer, sample_size=None):
    tokenizer.src_lang = lang_code
    if sample_size:
        texts = texts[:sample_size]
    lengths = [len(tokenizer(t).input_ids) for t in texts]
    lengths = np.array(lengths)
    return {
        "n": len(lengths),
        "mean": lengths.mean(),
        "median": np.median(lengths),
        "p95": np.percentile(lengths, 95),
        "max": lengths.max(),
    }

en_stats = token_length_stats(train_en_sample, "eng_Latn", tokenizer)
ar_stats = token_length_stats(train_ar_sample, "arb_Arab", tokenizer)

print("EN token-length stats:", en_stats)
print("AR token-length stats:", ar_stats)

EN token-length stats: {'n': 39997, 'mean': np.float64(16.746781008575642), 'median': np.float64(12.0), 'p95': np.float64(47.0), 'max': np.int64(179)}
AR token-length stats: {'n': 39997, 'mean': np.float64(19.754781608620647), 'median': np.float64(13.0), 'p95': np.float64(65.0), 'max': np.int64(309)}


In [41]:
import numpy as np

def token_length_stats(texts, lang_code, tokenizer, sample_size=None):
    tokenizer.src_lang = lang_code
    if sample_size:
        texts = texts[:sample_size]
    lengths = [len(tokenizer(t).input_ids) for t in texts]
    lengths = np.array(lengths)
    return {
        "n": len(lengths),
        "mean": lengths.mean(),
        "median": np.median(lengths),
        "p95": np.percentile(lengths, 95),
        "max": lengths.max(),
    }

en_stats = token_length_stats(train_en_sample, "eng_Latn", tokenizer)
ar_stats = token_length_stats(train_ar_sample, "arb_Arab", tokenizer)

print("EN token-length stats:", en_stats)
print("AR token-length stats:", ar_stats)

EN token-length stats: {'n': 39997, 'mean': np.float64(16.746781008575642), 'median': np.float64(12.0), 'p95': np.float64(47.0), 'max': np.int64(179)}
AR token-length stats: {'n': 39997, 'mean': np.float64(19.754781608620647), 'median': np.float64(13.0), 'p95': np.float64(65.0), 'max': np.int64(309)}


In [42]:
#
#from transformers import AutoModelForSeq2SeqLM
#
#model_name = "facebook/nllb-200-distilled-600M"
#model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

#print("Model loaded.")
#print("Num parameters:", sum(p.numel() for p in model.parameters()))


In [43]:
import nltk
from nltk.translate import IBMModel1, AlignedSent

print(nltk.__version__)
print(IBMModel1)

3.10.3
<class 'nltk.translate.ibm1.IBMModel1'>


In [44]:
bitext = [
    AlignedSent(en.split(), ar.split())
    for en, ar in zip(train_en_sample, train_ar_sample)
]

print(len(bitext))
print(bitext[0].words)      # English (source)
print(bitext[0].mots)       # Arabic (target) -- nltk calls target words "mots"

39997
['You', 'first.', "That's", 'nasty.']
['انت', 'اول', 'ياشقي']


In [45]:
import time

start = time.time()
ibm1 = IBMModel1(bitext, 5)  # 5 EM iterations (nltk default)
elapsed = time.time() - start

print(f"Training took {elapsed:.1f} seconds")

Training took 167.9 seconds


In [46]:
# Look at what Arabic words the model thinks are most likely for a few common English words
test_words = ["I", "you", "the", "love", "know"]

for w in test_words:
    if w in ibm1.translation_table:
        # Get top 5 most probable Arabic translations for this English word
        translations = sorted(
            ibm1.translation_table[w].items(),
            key=lambda x: -x[1]
        )[:5]
        print(f"'{w}' ->")
        for ar_word, prob in translations:
            print(f"    {ar_word!r}: {prob:.4f}")
    else:
        print(f"'{w}' not in vocabulary")
    print()

'I' ->
    'انني': 0.8032
    'I': 0.7653
    'أنني': 0.6553
    'أني': 0.6456
    'أنّي': 0.6323

'you' ->
    'you': 0.9864
    'أنك': 0.6468
    'بأنّك': 0.6444
    'لكم': 0.6403
    'هل': 0.6378

'the' ->
    'the': 0.8325
    'العرض': 0.6408
    'الباب': 0.5805
    'النهاية': 0.4724
    'الحدود': 0.4707

'love' ->
    'love': 0.5829
    'الحب': 0.5277
    'أحبها': 0.4643
    'أعشق': 0.4258
    '-أحب': 0.4201

'know' ->
    'تعرف': 0.6880
    'know': 0.6762
    'تعرفين': 0.6416
    'نعلم': 0.6389
    'اعلم': 0.5988



In [47]:
def translate_ibm1_monotone(en_sentence, model):
    en_words = en_sentence.split()
    translated = []
    for w in en_words:
        if w in model.translation_table and model.translation_table[w]:
            best_ar_word = max(model.translation_table[w].items(), key=lambda x: x[1])[0]
            translated.append(best_ar_word)
        else:
            translated.append(w)  # OOV -> pass through untouched
    return " ".join(translated)

# Quick test on a few train examples (not test set yet -- just sanity-checking the function)
for i in range(5):
    src = train_en_sample[i]
    ref = train_ar_sample[i]
    hyp = translate_ibm1_monotone(src, ibm1)
    print(f"EN:  {src}")
    print(f"REF: {ref}")
    print(f"HYP: {hyp}")
    print()

EN:  You first. That's nasty.
REF: انت اول ياشقي
HYP: إنّك أولاً. ...ذلك ياشقي

EN:  #You're not the boss of me now #
REF: ? أنتَلستالرئيسعليّبعدالآن?
HYP: أنتَلستالرئيسعليّبعدالآن? وليس the أشيري of لي والان للة

EN:  Sookie.
REF: ! سوكي
HYP: سوكي

EN:  - I could love you, you know. - Faith...
REF: يمكنني أن أحبك - ...(فايث) -
HYP: - انني أمكنك love لك، you اعرف- - ...(فايث)

EN:  I can take care of myself.
REF: أستطيع الاعتناء بنفسي
HYP: انني يُمْكِنُ تأخذ تهتم of بنفسي



In [48]:
import re

# Does this Arabic "sentence" contain any Latin letters at all?
def has_latin(text):
    return bool(re.search(r'[A-Za-z]', text))

contaminated = [i for i, ar in enumerate(train_ar_sample) if has_latin(ar)]
print(f"Arabic lines containing Latin script: {len(contaminated)} / {len(train_ar_sample)} ({100*len(contaminated)/len(train_ar_sample):.2f}%)")

# Show a handful of examples to see what it actually looks like
for i in contaminated[:8]:
    print(f"[{i}] EN: {train_en_sample[i]}")
    print(f"     AR: {train_ar_sample[i]}")
    print()

Arabic lines containing Latin script: 983 / 39997 (2.46%)
[86] EN: New opportunities for some Geneva-based commercial activities such as the Visitors' Service, bookshop and gift centre, including their increased synergies with similar activities by other Geneva-based organizations of the system, could arise from the project currently being studied by the Swiss authorities to physically transform and modernize the Place des Nations.
     AR: وهناك فرص جديدة متاحة لبعض الأنشطة التجارية التي يوجد مقرها في جنيف، مثل أنشطة خدمات الزوار والمكتبة ومركز بيع الهدايا، بما في ذلك زيادة أوجه تآزرها مع الأنشطة المماثلة التي تضطلع بها مؤسسات أخرى في المنظومة يوجد مقرها في جنيف، وهي فرص يمكن أن تنشأ عن المشروع الذي تعكف السلطات السويسرية على دراستــه حالياً فيما يتصل بإدخال تغييرات ماديـة علـى ساحـة الأمم (Place des Nations) وتحديثها.

[134] EN: Protocol to Prevent, Suppress and Punish Trafficking in Persons, Especially Women and Children, supplementing the United Nations Convention against Transnati

In [49]:
# 1. How many are near-exact duplicates (genuinely untranslated)?
exact_dupes = [i for i in contaminated if train_en_sample[i].strip() == train_ar_sample[i].strip()]
print(f"Exact EN==AR duplicates: {len(exact_dupes)}")

# 2. How many contain the explicit placeholder?
placeholder = [i for i in contaminated if "untranslated" in train_ar_sample[i].lower()]
print(f"Explicit 'untranslated' placeholder: {len(placeholder)}")

# 3. How many still contain real Arabic script alongside the Latin (likely benign proper-noun case)?
def has_arabic(text):
    return bool(re.search(r'[\u0600-\u06FF]', text))

mixed_with_arabic = [i for i in contaminated if has_arabic(train_ar_sample[i]) and i not in exact_dupes]
print(f"Contains Latin AND real Arabic script (likely benign): {len(mixed_with_arabic)}")

# 4. Remainder - Latin-only AR side, not an exact dupe (garbled/partial cases like [180], [252])
other = [i for i in contaminated if i not in exact_dupes and i not in mixed_with_arabic]
print(f"Other (Latin-only AR, not exact dupe): {len(other)}")

Exact EN==AR duplicates: 174
Explicit 'untranslated' placeholder: 1
Contains Latin AND real Arabic script (likely benign): 708
Other (Latin-only AR, not exact dupe): 101


In [50]:
def latin_word_ratio(text):
    words = text.split()
    if not words:
        return 0
    latin_words = [w for w in words if re.search(r'[A-Za-z]', w)]
    return len(latin_words) / len(words)

ratios = [(i, latin_word_ratio(train_ar_sample[i])) for i in mixed_with_arabic]
ratio_vals = np.array([r for _, r in ratios])

print(f"n = {len(ratios)}")
print(f"mean={ratio_vals.mean():.3f}  median={np.median(ratio_vals):.3f}")
for thresh in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    print(f"  ratio >= {thresh}: {(ratio_vals >= thresh).sum()} lines")

# Show a few examples at the extremes so we can eyeball what each end actually looks like
high_ratio = sorted(ratios, key=lambda x: -x[1])[:5]
low_ratio = sorted(ratios, key=lambda x: x[1])[:5]

print("\n--- HIGH latin ratio (likely half-translation) ---")
for i, r in high_ratio:
    print(f"[{i}] ratio={r:.2f}")
    print(f"  EN: {train_en_sample[i]}")
    print(f"  AR: {train_ar_sample[i]}")

print("\n--- LOW latin ratio (likely proper noun) ---")
for i, r in low_ratio:
    print(f"[{i}] ratio={r:.2f}")
    print(f"  EN: {train_en_sample[i]}")
    print(f"  AR: {train_ar_sample[i]}")

n = 708
mean=0.187  median=0.116
  ratio >= 0.1: 403 lines
  ratio >= 0.2: 241 lines
  ratio >= 0.3: 142 lines
  ratio >= 0.5: 74 lines
  ratio >= 0.7: 22 lines
  ratio >= 0.9: 10 lines

--- HIGH latin ratio (likely half-translation) ---
[1441] ratio=1.00
  EN: - Hup, hup, hup, hup.
  AR: -Hup، HUP، HUP، HUP.
[2948] ratio=1.00
  EN: Notes
  AR: ملاحظاتComment
[4670] ratio=1.00
  EN: Goken?
  AR: Goken؟
[20761] ratio=1.00
  EN: 1), A/60/6 (Sect.
  AR: 1)،A/60/6 (Sect.
[23545] ratio=1.00
  EN: It's not a phone.
  AR: هولَيسَaهاتف.

--- LOW latin ratio (likely proper noun) ---
[22164] ratio=0.01
  EN: Mindful that, in its resolution 44/1 of 17 February 2006,Ibid., 2006, Supplement No. 6 (E/2006/26), chap. I, sect. C. the Commission for Social Development endorsed the calendar and the global theme for the first review and appraisal of the implementation of the Madrid Plan of Action, "Addressing the challenges and opportunities of ageing", and decided to start the first global cycle of revi

In [51]:
RATIO_THRESHOLD = 0.5
high_ratio_drop = [i for i, r in ratios if r >= RATIO_THRESHOLD]
print(f"Mixed lines dropped (ratio >= {RATIO_THRESHOLD}): {len(high_ratio_drop)}")

drop_indices = set(exact_dupes) | set(other) | set(high_ratio_drop)
print(f"Total lines to drop: {len(drop_indices)} ({100*len(drop_indices)/len(train_en_sample):.2f}%)")

keep_indices = [i for i in range(len(train_en_sample)) if i not in drop_indices]
train_en_sample = [train_en_sample[i] for i in keep_indices]
train_ar_sample = [train_ar_sample[i] for i in keep_indices]

print(f"New train subsample size: {len(train_en_sample)} pairs")

# Rebuild bitext and retrain IBM Model 1 on the cleaned data
bitext = [AlignedSent(en.split(), ar.split()) for en, ar in zip(train_en_sample, train_ar_sample)]

start = time.time()
ibm1 = IBMModel1(bitext, 5)
elapsed = time.time() - start
print(f"Retraining took {elapsed:.1f} seconds")

Mixed lines dropped (ratio >= 0.5): 74
Total lines to drop: 349 (0.87%)
New train subsample size: 39648 pairs
Retraining took 158.9 seconds


In [52]:
test_words = ["I", "you", "the", "love", "know"]

for w in test_words:
    if w in ibm1.translation_table:
        translations = sorted(
            ibm1.translation_table[w].items(),
            key=lambda x: -x[1]
        )[:5]
        print(f"'{w}' ->")
        for ar_word, prob in translations:
            print(f"    {ar_word!r}: {prob:.4f}")
    else:
        print(f"'{w}' not in vocabulary")
    print()

def translate_ibm1_monotone(en_sentence, model):
    en_words = en_sentence.split()
    translated = []
    for w in en_words:
        if w in model.translation_table and model.translation_table[w]:
            best_ar_word = max(model.translation_table[w].items(), key=lambda x: x[1])[0]
            translated.append(best_ar_word)
        else:
            translated.append(w)  # OOV -> pass through untouched
    return " ".join(translated)

# Quick test on a few train examples (not test set yet -- just sanity-checking the function)
for i in range(5):
    src = train_en_sample[i]
    ref = train_ar_sample[i]
    hyp = translate_ibm1_monotone(src, ibm1)
    print(f"EN:  {src}")
    print(f"REF: {ref}")
    print(f"HYP: {hyp}")
    print()

'I' ->
    'انني': 0.8031
    'أنني': 0.6552
    'أني': 0.6455
    'أنّي': 0.6321
    'اني': 0.6281

'you' ->
    'أنك': 0.6468
    'بأنّك': 0.6445
    'لكم': 0.6404
    'هل': 0.6381
    'هَلْ': 0.6347

'the' ->
    'العرض': 0.6409
    'الباب': 0.5807
    'النهاية': 0.4722
    'الحدود': 0.4711
    'الأشعة': 0.4690

'love' ->
    'الحب': 0.5276
    'أحبها': 0.4644
    'أعشق': 0.4259
    '-أحب': 0.4203
    'يحبّون': 0.4119

'know' ->
    'تعرف': 0.6881
    'تعرفين': 0.6417
    'نعلم': 0.6389
    'اعلم': 0.5992
    'وأعلم': 0.5928

EN:  You first. That's nasty.
REF: انت اول ياشقي
HYP: إنّك أولاً. ...ذلك ياشقي

EN:  #You're not the boss of me now #
REF: ? أنتَلستالرئيسعليّبعدالآن?
HYP: أنتَلستالرئيسعليّبعدالآن? وليس العرض أشيري صور لي والان للة

EN:  Sookie.
REF: ! سوكي
HYP: سوكي

EN:  - I could love you, you know. - Faith...
REF: يمكنني أن أحبك - ...(فايث) -
HYP: - انني أمكنك الحب لك، أنك اعرف- - ...(فايث)

EN:  I can take care of myself.
REF: أستطيع الاعتناء بنفسي
HYP: انني يُمْكِنُ تأخذ

In [53]:
def translate_monotone(sentence, translation_table, oov_token_passthrough=True):
    """
    Translate an English sentence to Arabic using IBM Model 1,
    emitting translated words in the same order as the source (monotone/no reordering).
    """
    words = sentence.split()
    translated = []
    for w in words:
        if w in translation_table and translation_table[w]:
            # Pick the Arabic word with highest t(ar|en) probability
            best_ar = max(translation_table[w].items(), key=lambda x: x[1])[0]
            translated.append(best_ar)
        else:
            # OOV: word never seen in training -> pass through untranslated
            # (a legitimate, nameable baseline failure mode, not a bug)
            if oov_token_passthrough:
                translated.append(w)
    return " ".join(translated)

# Quick smoke test on a few train sentences (not evaluation yet, just sanity-check output shape)
for i in [0, 1, 4]:
    src = train_en_sample[i]
    out = translate_monotone(src, ibm1.translation_table)
    print(f"EN:  {src}")
    print(f"OUT: {out}")
    print(f"REF: {train_ar_sample[i]}")
    print()

EN:  You first. That's nasty.
OUT: إنّك أولاً. ...ذلك ياشقي
REF: انت اول ياشقي

EN:  #You're not the boss of me now #
OUT: أنتَلستالرئيسعليّبعدالآن? وليس العرض أشيري صور لي والان للة
REF: ? أنتَلستالرئيسعليّبعدالآن?

EN:  I can take care of myself.
OUT: انني يُمْكِنُ تأخذ تهتم صور بنفسي
REF: أستطيع الاعتناء بنفسي



In [54]:
import sacrebleu

# Generate monotone translations for the entire test set
test_predictions = [translate_monotone(en, ibm1.translation_table) for en in test_en]

# Corpus-level BLEU and chrF against references
bleu = sacrebleu.corpus_bleu(test_predictions, [test_ar])
chrf = sacrebleu.corpus_chrf(test_predictions, [test_ar])

print(f"IBM Model 1 (monotone) baseline — test set ({len(test_en)} pairs)")
print(f"BLEU: {bleu.score:.2f}")
print(f"chrF: {chrf.score:.2f}")

IBM Model 1 (monotone) baseline — test set (2000 pairs)
BLEU: 2.78
chrF: 23.75


In [55]:
import random
random.seed(42)
sample_idx = random.sample(range(len(test_en)), 5)

ibm1_examples = []
for i in sample_idx:
    ex = {
        "en": test_en[i],
        "ar_ref": test_ar[i],
        "ibm1_pred": test_predictions[i],
    }
    ibm1_examples.append(ex)
    print(f"EN:   {ex['en']}")
    print(f"REF:  {ex['ar_ref']}")
    print(f"IBM1: {ex['ibm1_pred']}")
    print()

EN:   "I know I'll feel better after I do this".
REF:  "اعلم اننى سوف اشعر بتحسن بعدما انهى هذا".
IBM1: "أنا تعرف سأتذكر أشعر الأفضل بعد انني تفعلي this".

EN:   - I like to check the table for the phone. If you don't mind.
REF:  اريد ان اتفقد الطاوله بحثا عن الهاتف بعد اذنك
IBM1: - انني أشبه لكي إسمحوا العرض الجدول لأجل العرض الهاتف. -إن أنك ر تبالي

EN:   He asked me details with a gun to my head.
REF:  سألني لمزيد من التفاصيل. بوينت بلانك.
IBM1: كاد طَلبَ لي التفاصيل مَع a بندقيتك لكي بلدي ضربتك

EN:   Boris.
REF:  (بوريس)
IBM1: وستعمل

EN:   Arthur Bailey.
REF:  آرثر بيلي.
IBM1: (أرثر) ترين،



In [56]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from huggingface_hub import scan_cache_dir
import time

model_name = "facebook/nllb-200-distilled-600M"

# Check HF cache before loading anything
cache_info = scan_cache_dir()
cached_models = [repo.repo_id for repo in cache_info.repos]

if model_name in cached_models:
    print(f"'{model_name}' already cached locally -- will load from disk, no download.")
else:
    print(f"'{model_name}' NOT cached yet -- this will download ~2.4GB. May take a few minutes depending on connection.")

# Tokenizer is already cached from Day 2's sanity check -- this call is instant
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# This is the actual model weights load (or download if not cached)
print("Loading model (this is the slow part if not cached)...")
start = time.time()
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()  # inference mode, disables dropout etc.
elapsed = time.time() - start
print(f"Model loaded in {elapsed:.1f}s")

'facebook/nllb-200-distilled-600M' already cached locally -- will load from disk, no download.

Loading tokenizer...
Loading model (this is the slow part if not cached)...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Model loaded in 8.5s


In [57]:
def translate_batch_nllb(sentences, model, tokenizer, src_lang="eng_Latn", tgt_lang="arb_Arab", max_length=128):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=max_length)

    # Force decoder to start with the target language token
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        num_beams=1,  # greedy decoding -- beam search is Day 7's experiment, not here
    )

    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)


# Smoke test on 10 sentences
smoke_sentences = test_en[:10]

start = time.time()
smoke_translations = translate_batch_nllb(smoke_sentences, model, tokenizer)
elapsed = time.time() - start

print(f"Translated {len(smoke_sentences)} sentences in {elapsed:.1f}s ({elapsed/len(smoke_sentences):.2f}s/sentence)\n")

for en, ar_ref, ar_pred in zip(smoke_sentences, test_ar[:10], smoke_translations):
    print(f"EN:   {en}")
    print(f"REF:  {ar_ref}")
    print(f"PRED: {ar_pred}")
    print()

Translated 10 sentences in 31.5s (3.15s/sentence)

EN:   Well, according to the report, there were three water bottles missing.
REF:  حسناً ، تبعاً للتقرير هناك ثلاث عبوات ماء مفقودة
PRED: حسناً، وفقاً للتقرير، فقدت ثلاثة زجاجات ماء

EN:   The Commission reviewed the data on recruitment and retention difficulties, which it had requested from the organizations.
REF:  81 - استعرضت اللجنة البيانات التي طلبتها من المؤسسات عن صعوبات التوظيف والاستبقاء.
PRED: استعرضت المفوضية بيانات صعوبات التوظيف والاحتفاظ بها التي طلبتها من المنظمات.

EN:   (Maya) So Are You Gonna Move Back In,
REF:  هل سترجع للعيش معنا هنا
PRED: (مايا) إذاً هل ستعودين إلى هنا

EN:   Yes, you may.
REF:  أجل، تستطيع
PRED: نعم، يمكنك.

EN:   What are you scared of?
REF:  مما أنت خائف؟
PRED: ما الذي تخافين منه؟

EN:   Like my husband always says...
REF:  : كما يقول زوجي دائماً
PRED: كما يقول زوجي دائماً

EN:   Come on, muscle relaxers.
REF:  هيّا يا مهدّئات العضلات.
PRED: هيا، أخصائيات العضلات

EN:   United Nations Juridical 

In [58]:
import torch, os

print(f"CPU cores available: {os.cpu_count()}")
print(f"PyTorch threads (current): {torch.get_num_threads()}")

torch.set_num_threads(os.cpu_count())
print(f"PyTorch threads (set to max): {torch.get_num_threads()}")

# Try a larger batch and measure throughput
bigger_batch = test_en[:32]

start = time.time()
bigger_translations = translate_batch_nllb(bigger_batch, model, tokenizer)
elapsed = time.time() - start

print(f"\nTranslated {len(bigger_batch)} sentences in {elapsed:.1f}s ({elapsed/len(bigger_batch):.2f}s/sentence)")

CPU cores available: 12
PyTorch threads (current): 6
PyTorch threads (set to max): 12

Translated 32 sentences in 70.9s (2.22s/sentence)


In [59]:
random.seed(42)
subsample_idx = sorted(random.sample(range(len(test_en)), 200))

test_en_sub = [test_en[i] for i in subsample_idx]
test_ar_sub = [test_ar[i] for i in subsample_idx]

print(f"Subsample size: {len(test_en_sub)}")

# Batched generation with progress tracking, since this will take a few minutes
BATCH_SIZE = 32
nllb_predictions_sub = []

start = time.time()
for i in range(0, len(test_en_sub), BATCH_SIZE):
    batch = test_en_sub[i:i+BATCH_SIZE]
    translations = translate_batch_nllb(batch, model, tokenizer)
    nllb_predictions_sub.extend(translations)
    print(f"  {min(i+BATCH_SIZE, len(test_en_sub))}/{len(test_en_sub)} done "
          f"({time.time()-start:.0f}s elapsed)")

total_elapsed = time.time() - start
print(f"\nTotal: {total_elapsed:.1f}s for {len(test_en_sub)} sentences "
      f"({total_elapsed/len(test_en_sub):.2f}s/sentence)")

Subsample size: 200
  32/200 done (329s elapsed)
  64/200 done (635s elapsed)
  96/200 done (811s elapsed)
  128/200 done (987s elapsed)
  160/200 done (1202s elapsed)
  192/200 done (1523s elapsed)
  200/200 done (1561s elapsed)

Total: 1560.5s for 200 sentences (7.80s/sentence)


In [60]:
# NLLB pretrained scores on subsample
nllb_bleu = sacrebleu.corpus_bleu(nllb_predictions_sub, [test_ar_sub])
nllb_chrf = sacrebleu.corpus_chrf(nllb_predictions_sub, [test_ar_sub])

# Re-score IBM Model 1 on the SAME 200 indices for fair comparison
ibm1_predictions_sub = [translate_monotone(en, ibm1.translation_table) for en in test_en_sub]
ibm1_bleu_sub = sacrebleu.corpus_bleu(ibm1_predictions_sub, [test_ar_sub])
ibm1_chrf_sub = sacrebleu.corpus_chrf(ibm1_predictions_sub, [test_ar_sub])

print(f"=== Results on n=200 subsample ===\n")
print(f"IBM Model 1 (monotone):")
print(f"  BLEU: {ibm1_bleu_sub.score:.2f}   chrF: {ibm1_chrf_sub.score:.2f}")
print(f"\nNLLB-200-distilled-600M (pretrained, no fine-tuning):")
print(f"  BLEU: {nllb_bleu.score:.2f}   chrF: {nllb_chrf.score:.2f}")
print(f"  Inference: {total_elapsed:.1f}s for 200 sentences ({total_elapsed/200:.2f}s/sentence avg)")

print(f"\n=== For reference: IBM Model 1 on full 2000-sentence test set ===")
print(f"  BLEU: 2.78   chrF: 23.75")

=== Results on n=200 subsample ===

IBM Model 1 (monotone):
  BLEU: 2.26   chrF: 23.56

NLLB-200-distilled-600M (pretrained, no fine-tuning):
  BLEU: 13.54   chrF: 44.96
  Inference: 1560.5s for 200 sentences (7.80s/sentence avg)

=== For reference: IBM Model 1 on full 2000-sentence test set ===
  BLEU: 2.78   chrF: 23.75


In [61]:
random.seed(42)
example_idx_in_sub = random.sample(range(len(test_en_sub)), 5)

nllb_examples = []
for i in example_idx_in_sub:
    ex = {
        "en": test_en_sub[i],
        "ar_ref": test_ar_sub[i],
        "nllb_pred": nllb_predictions_sub[i],
    }
    nllb_examples.append(ex)
    print(f"EN:   {ex['en']}")
    print(f"REF:  {ex['ar_ref']}")
    print(f"NLLB: {ex['nllb_pred']}")
    print()

EN:   It don't matter a damn. I was just curious, that's all.
REF:  هذا ليس من شأني البتة، فقد إنتابني الفضول وحسب، هذا كل ما في الأمر
NLLB: لا يهمّ، كنتُ فضولية فحسب، هذا كلّ شيء

EN:   Absolute museum-quality.
REF:  نوعية تناسب المتاحف
NLLB: نوعية متحفية مطلقة

EN:   Especially not right now...
REF:  وخصوصا ليس الآن...
NLLB: خاصة الآن

EN:   My father has a ranch on the outskirts of town.
REF:  لدى أبي مزرعة على مشارف القرية
NLLB: أبي لديه مزرعة في ضواحي المدينة

EN:   - Rizza will...
REF:  - . (ريـزّا) .
NLLB: - ريزا سوف ...



In [62]:
for en, ar_ref, ar_pred in zip(smoke_sentences[:5], test_ar[:5], smoke_translations[:5]):
    sent_bleu = sacrebleu.sentence_bleu(ar_pred, [ar_ref])
    print(f"EN:   {en}")
    print(f"REF:  {ar_ref}")
    print(f"PRED: {ar_pred}")
    print(f"Sentence BLEU: {sent_bleu.score:.2f}")
    print()

EN:   Well, according to the report, there were three water bottles missing.
REF:  حسناً ، تبعاً للتقرير هناك ثلاث عبوات ماء مفقودة
PRED: حسناً، وفقاً للتقرير، فقدت ثلاثة زجاجات ماء
Sentence BLEU: 4.94

EN:   The Commission reviewed the data on recruitment and retention difficulties, which it had requested from the organizations.
REF:  81 - استعرضت اللجنة البيانات التي طلبتها من المؤسسات عن صعوبات التوظيف والاستبقاء.
PRED: استعرضت المفوضية بيانات صعوبات التوظيف والاحتفاظ بها التي طلبتها من المنظمات.
Sentence BLEU: 14.60

EN:   (Maya) So Are You Gonna Move Back In,
REF:  هل سترجع للعيش معنا هنا
PRED: (مايا) إذاً هل ستعودين إلى هنا
Sentence BLEU: 6.57

EN:   Yes, you may.
REF:  أجل، تستطيع
PRED: نعم، يمكنك.
Sentence BLEU: 0.00

EN:   What are you scared of?
REF:  مما أنت خائف؟
PRED: ما الذي تخافين منه؟
Sentence BLEU: 0.00



In [63]:
import random

random.seed(42)
tiny_indices = random.sample(range(len(train_en_sample)), 8)

tiny_en = [train_en_sample[i] for i in tiny_indices]
tiny_ar = [train_ar_sample[i] for i in tiny_indices]

for en, ar in zip(tiny_en, tiny_ar):
    print("EN:", en)
    print("AR:", ar)
    print("---")

EN: - $2,000. They say the clothes make the man.
AR: يقولون أن الملابس تصنع الرّجل
---
EN: I'll be seeing ya.
AR: سأراك قريبًا
---
EN: So do I get a hotel?
AR: إذاً، هل سأحصل على غرفة في فندق؟
---
EN: Under agenda item 14 as a whole, the Council had before it the following documents:
AR: 210 - وفي إطار البند 14 من جدول الأعمال ككل، كان معروضا على المجلس الوثيقتان التاليتان:
---
EN: It's a little early, isn't it?
AR: لا يزال الوقت مبكرا, أليس كذلك؟
---
EN: Through its training programme, UNMOVIC has established a roster of inspectors comprising over 350 experts in the biological, chemical and missile areas drawn from more than 50 countries.
AR: وأعدت أنموفيك، عبر برنامجها التدريبي، قائمة بالمفتشين شملت أكثر من 350 خبيرا في المجالات البيولوجية والكيميائية ومجال الصواريخ، جرى اختيارهم من أكثر من 50 بلدا.
---
EN: One of the first terrorist acts committed against merchant vessels carrying cargo bound for Cuba was the above-mentioned attack on the French steamship, La Coubre, on 4 March 1960

In [64]:
import re

un_keywords_en = [
    "united nations", "security council", "general assembly", "secretary-general",
    "agenda item", "pursuant to", "resolution", "member states", "committee",
    "report of the", "in accordance with", "annex", "paragraph"
]

# leading paragraph numbering, e.g. "210 - " or "14. " at start of line
leading_number_pattern = re.compile(r"^\s*\d+\s*[-.]\s")

def is_un_style(en_line, ar_line):
    en_lower = en_line.lower()
    if any(kw in en_lower for kw in un_keywords_en):
        return True
    if leading_number_pattern.match(en_line) or leading_number_pattern.match(ar_line):
        return True
    return False

flags = [is_un_style(en, ar) for en, ar in zip(train_en_sample, train_ar_sample)]
n_flagged = sum(flags)
pct = 100 * n_flagged / len(train_en_sample)

print(f"Flagged as UN/formal-style: {n_flagged} / {len(train_en_sample)} ({pct:.2f}%)")

flagged_examples = [(en, ar) for en, ar, f in zip(train_en_sample, train_ar_sample, flags) if f][:10]
for en, ar in flagged_examples:
    print("EN:", en)
    print("AR:", ar)
    print("---")

Flagged as UN/formal-style: 2531 / 39648 (6.38%)
EN: The sixth and seventh sessions of the Committee of Experts on International Cooperation in Tax Matters are scheduled to be held in 2010 and 2011, respectively.
AR: ومن المقرر أن تعقد الدورتان السادسة والسابعة للجنة الخبراء المعنية بالتعاون الدولي في المسائل الضريبية في عامي 2010 و 2011 على التوالي.
---
EN: In answering this question, State are encouraged to address, among other issues, the following specific questions:
AR: 3- وتشجع الدول عند الإجابة عن هذا السؤال على تناول أمور من بينها الأسئلة المحددة التالية:
---
EN: Faced with the choice between, on the one hand, a method not involving a rapporteur and enabling the preliminary report to be submitted to the Commission in 2001 but not, to its way of thinking, providing adequate guarantees (in an extremely limited timeframe) of scholarly diligence or in-depth analysis of the issue and, on the other hand, a method it considered the most appropriate (the rapporteur) and the scrupulous 

In [65]:
test_flags = [is_un_style(en, ar) for en, ar in zip(test_en, test_ar)]
n_test_flagged = sum(test_flags)
print(f"Test set flagged as UN/formal-style: {n_test_flagged} / {len(test_en)} ({100*n_test_flagged/len(test_en):.2f}%)")

Test set flagged as UN/formal-style: 138 / 2000 (6.90%)


In [66]:
!pip install scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# reuse the seeded 200-sentence test subsample from Day 3 (test_en_sub)
vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2))
train_vectors = vectorizer.fit_transform(train_en_sample)   # 39,648 x vocab
test_vectors = vectorizer.transform(test_en_sub)             # 200 x vocab

sims = cosine_similarity(test_vectors, train_vectors)        # 200 x 39,648
best_idx = sims.argmax(axis=1)
best_sim = sims.max(axis=1)

print("Best-match TF-IDF cosine similarity over 200 test sentences:")
print(f"  mean:   {best_sim.mean():.3f}")
print(f"  median: {np.median(best_sim):.3f}")
print(f"  % above 0.7 (strong/near-duplicate match): {100*(best_sim > 0.7).mean():.1f}%")
print(f"  % above 0.5 (moderate match):              {100*(best_sim > 0.5).mean():.1f}%")
print(f"  % above 0.3 (weak match):                  {100*(best_sim > 0.3).mean():.1f}%")

print("\n5 example top matches:")
for i in np.random.RandomState(42).choice(len(test_en_sub), 5, replace=False):
    print(f"TEST EN : {test_en_sub[i]}")
    print(f"MATCH EN: {train_en_sample[best_idx[i]]}  (sim={best_sim[i]:.3f})")
    print(f"MATCH AR: {train_ar_sample[best_idx[i]]}")
    print("---")


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Best-match TF-IDF cosine similarity over 200 test sentences:
  mean:   0.454
  median: 0.419
  % above 0.7 (strong/near-duplicate match): 11.0%
  % above 0.5 (moderate match):              31.5%
  % above 0.3 (weak match):                  79.5%

5 example top matches:
TEST EN : Michael is just...
MATCH EN: Michael!  (sim=0.634)
MATCH AR: ! مايكل
---
TEST EN : I'm choking on the stuff.
MATCH EN: It's not about the stuff.  (sim=0.453)
MATCH AR: أنه ليس حول الأغراض
---
TEST EN : Relaxing your body...
MATCH EN: All over your body.  (sim=0.675)
MATCH AR: على كل جسده
---
TEST EN : Members of permanent missions, Secretary staff, representatives of non-governmental organizations and the media are invited to attend.
MATCH EN: (d) To encourage the involvement of non-governmental organizations in the dissemination of information on decolonization;  (sim=0.227)
MATCH AR: (د) تشجيع مشاركة المنظمات غير الحكومية في نشر المعلومات المتعلقة بإنهاء الاستعمار؛
---
TEST EN : "I know I'll feel better after

In [67]:
!pip install spacy
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta 


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [68]:
import spacy
import sacrebleu
import numpy as np

nlp = spacy.load("en_core_web_sm")

# Tag each test sentence: does it contain a PERSON, ORG, GPE, or NORP entity?
# (these are the categories most likely to need transliteration, not translation)
relevant_types = {"PERSON", "ORG", "GPE", "NORP", "FAC", "LOC"}
has_entity = []
entity_details = []

for doc in nlp.pipe(test_en_sub):
    ents = [(e.text, e.label_) for e in doc.ents if e.label_ in relevant_types]
    has_entity.append(len(ents) > 0)
    entity_details.append(ents)

has_entity = np.array(has_entity)
print(f"Sentences with a relevant named entity: {has_entity.sum()} / {len(test_en_sub)} ({100*has_entity.mean():.1f}%)")

# Per-sentence chrF for NLLB predictions, split by entity presence
chrf = sacrebleu.CHRF()
per_sentence_chrf = np.array([
    chrf.sentence_score(pred, [ref]).score
    for pred, ref in zip(nllb_predictions_sub, test_ar_sub)
])

print(f"\nNLLB chrF — sentences WITH entity:    mean={per_sentence_chrf[has_entity].mean():.2f}, n={has_entity.sum()}")
print(f"NLLB chrF — sentences WITHOUT entity: mean={per_sentence_chrf[~has_entity].mean():.2f}, n={(~has_entity).sum()}")

# A few concrete examples where an entity is present, to eyeball actual failure modes
print("\n5 example entity-containing sentences with predictions:")
entity_indices = np.where(has_entity)[0]
for i in np.random.RandomState(42).choice(entity_indices, min(5, len(entity_indices)), replace=False):
    print(f"EN:      {test_en_sub[i]}")
    print(f"Entities: {entity_details[i]}")
    print(f"REF AR:  {test_ar_sub[i]}")
    print(f"PRED AR: {nllb_predictions_sub[i]}")
    print(f"chrF: {per_sentence_chrf[i]:.2f}")
    print("---")

Sentences with a relevant named entity: 50 / 200 (25.0%)

NLLB chrF — sentences WITH entity:    mean=46.90, n=50
NLLB chrF — sentences WITHOUT entity: mean=37.98, n=150

5 example entity-containing sentences with predictions:
EN:      No, no, no. Jonas explained it.
Entities: [('Jonas', 'PERSON')]
REF AR:  لا , لا , لا , جوناس شرحها
PRED AR: لا، لا، لا، لا (جوناس) شرح ذلك
chrF: 27.10
---
EN:      8 See Official Records of the General Assembly, Fiftieth Session, Supplement No. 30 (A/50/30), paras. 172 (a) (ii) and (b) (ii).
Entities: [('the General Assembly', 'ORG')]
REF AR:  (8) انظر الوثائق الرسمية للجمعية العامة، الدورة الخمسون، الملحق رقم 30 A/50/50))، الفقرتان 172 (أ) '2` (ب) '2`.
PRED AR: 8 انظر السجلات الرسمية للجمعية العامة، الدورة الخامسة عشرة، الملحق رقم 30 (A/50/30) ، الفقرة 172 (أ) (ii) و (ب) (ii).
chrF: 58.90
---
EN:      In 2005, the Government presented a report on its progress towards the implementation of the Millennium Development Goals, which indicated that the povert

In [69]:
import numpy as np

lengths_en = np.array([len(s.split()) for s in test_en_sub])

print(f"Mean EN length — entity sentences:     {lengths_en[has_entity].mean():.1f} words")
print(f"Mean EN length — non-entity sentences: {lengths_en[~has_entity].mean():.1f} words")
print(f"Correlation (has_entity, length): {np.corrcoef(has_entity.astype(int), lengths_en)[0,1]:.3f}")
print(f"Correlation (length, chrF):       {np.corrcoef(lengths_en, per_sentence_chrf)[0,1]:.3f}")

# Simple OLS: chrF ~ length + has_entity, to isolate entity's effect after controlling for length
X = np.column_stack([np.ones(len(lengths_en)), lengths_en, has_entity.astype(int)])
y = per_sentence_chrf
coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)
print(f"\nOLS: chrF ≈ {coeffs[0]:.2f} + {coeffs[1]:.3f}·length + {coeffs[2]:.3f}·has_entity")
print(f"Entity effect after controlling for length: {coeffs[2]:+.2f} chrF points")

Mean EN length — entity sentences:     15.1 words
Mean EN length — non-entity sentences: 8.8 words
Correlation (has_entity, length): 0.260
Correlation (length, chrF):       0.176

OLS: chrF ≈ 35.61 + 0.270·length + 7.210·has_entity
Entity effect after controlling for length: +7.21 chrF points


In [70]:
import numpy as np

# Sort the 200-subsample by chrF, ascending — worst translations first
order = np.argsort(per_sentence_chrf)
worst_n = 20

print(f"{worst_n} WORST-scoring NLLB translations (by chrF):\n")
for i in order[:worst_n]:
    print(f"EN:   {test_en_sub[i]}")
    print(f"REF:  {test_ar_sub[i]}")
    print(f"PRED: {nllb_predictions_sub[i]}")
    print(f"chrF: {per_sentence_chrf[i]:.2f}  |  len={len(test_en_sub[i].split())}  |  has_entity={has_entity[i]}")
    print("---")

20 WORST-scoring NLLB translations (by chrF):

EN:   Dave--Davey--Dave.
REF:  Dave--Davey--Dave.
PRED: (ديف) ، (ديفي)
chrF: 0.00  |  len=1  |  has_entity=True
---
EN:   I really like you.
REF:  19: 08,810 -
PRED: أنا حقاً أحبك
chrF: 0.00  |  len=4  |  has_entity=False
---
EN:   Nausicaa.
REF:  ناوسكا
PRED: -المرض
chrF: 2.78  |  len=1  |  has_entity=False
---
EN:   - Sure, sure.
REF:  كنّا نمزح
PRED: -بالطبع، بالتأكيد
chrF: 3.47  |  len=3  |  has_entity=False
---
EN:   You did not just say that to me.
REF:  بحقكِ، لابد أن تخبريني بأي شيء!
PRED: لم تقل هذا لي
chrF: 3.65  |  len=8  |  has_entity=False
---
EN:   I'm choking on the stuff.
REF:  انه يتكون في جسمك
PRED: أنا أَخْنقُ على الأشياءِ.
chrF: 4.27  |  len=5  |  has_entity=False
---
EN:   Up here!
REF:  في الأعلى!
PRED: هنا!
chrF: 6.25  |  len=2  |  has_entity=False
---
EN:   We're here, Jules.
REF:  لقد وصلنا يا عزيزتي
PRED: نحن هنا، جولز.
chrF: 6.66  |  len=3  |  has_entity=True
---
EN:   Yo, pussy!
REF:  يو، كس!
PRED: يا، العاهرة!


In [71]:
# Set source and target language codes for the NLLB tokenizer
tokenizer.src_lang = "eng_Latn"
tokenizer.tgt_lang = "arb_Arab"

# Tokenize the tiny 8-pair verification set
tiny_inputs = tokenizer(
    tiny_en,
    text_target=tiny_ar,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128,
)

print("keys:", tiny_inputs.keys())
print("input_ids shape:", tiny_inputs["input_ids"].shape)
print("labels shape:", tiny_inputs["labels"].shape)

# Sanity check pair 0 — decode WITHOUT skipping special tokens so
# we can actually see the language code tokens sitting in the sequence
print("\n--- pair 0 ---")
print("EN source:", tiny_en[0])
print("input_ids[0] tokens:", tokenizer.convert_ids_to_tokens(tiny_inputs["input_ids"][0]))
print()
print("AR target:", tiny_ar[0])
print("labels[0] tokens:", tokenizer.convert_ids_to_tokens(tiny_inputs["labels"][0]))

keys: KeysView({'input_ids': tensor([[256047,     89,   4589,   4437,  90549,  13710,   2852,    349, 176516,
           7038,    349,    492, 248075,      2,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1],
        [256047,    117, 248116,   1015,    280, 140505,    160, 248075,      2,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1],
        [256047,   2025,    359,    117,   4023,      9,  21609, 248130,      2,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,     

In [72]:
import torch

# Copy labels so we don't mutate tiny_inputs in place unexpectedly
labels_for_loss = tiny_inputs["labels"].clone()
labels_for_loss[labels_for_loss == tokenizer.pad_token_id] = -100

# Sanity check: pair 0 should now show -100 where <pad> was
print(labels_for_loss[0])

tensor([256011,  93138,    332,   2899,  30728, 223109,  10363, 112567,   6051,
        248241,  13793,      2,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100])


In [ ]:
# Apply the -100 masking fix to the FULL labels tensor (not just row 0)
labels_for_loss = tiny_inputs["labels"].clone()
labels_for_loss[labels_for_loss == tokenizer.pad_token_id] = -100

# Forward pass on the tiny 8-pair set
model.train()  # not eval — we want this behaving like it will during real training
outputs = model(
    input_ids=tiny_inputs["input_ids"],
    attention_mask=tiny_inputs["attention_mask"],
    labels=labels_for_loss,
)

loss = outputs.loss
print("loss:", loss.item())
print("loss is finite:", torch.isfinite(loss).item())
print("logits shape:", outputs.logits.shape)

loss: 1.8954713344573975
loss is finite: True
logits shape: torch.Size([8, 65, 256206])


: 

In [ ]:
# Backward pass — populate gradients
loss.backward()

# Spot-check that gradients actually landed somewhere real,
# not just on an embedding table by coincidence
some_grad_found = False
for name, param in model.named_parameters():
    if param.grad is not None and param.grad.abs().sum().item() > 0:
        some_grad_found = True
        print("example grad found on:", name, "| grad abs sum:", param.grad.abs().sum().item())
        break

print("\nany gradient found at all:", some_grad_found)

# Also check total param count with a nonzero grad, for a fuller picture
n_with_grad = sum(1 for p in model.parameters() if p.grad is not None and p.grad.abs().sum().item() > 0)
n_total = sum(1 for p in model.parameters())
print(f"{n_with_grad}/{n_total} parameters received a nonzero gradient")